In [108]:
import numpy as np
import pandas as pd 

In [109]:
df = pd.read_csv(r'D:\Mohamed Refaat\DEPI\Project implemention\Intelligent-Support-Ticket\Project Implementation\Data\clean_data.csv')

In [110]:
df.head()

,created_at,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,platform,region,date
0,2024-01-31T05:14:27,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27
1,2024-10-20T06:15:49,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is reviewing this and will follow up within 24 hours.,Ticket closed without further action after no response from the customer.,238.32,0,neutral,3,web,other,2024-10-20 06:15:49
2,2023-08-27T16:08:33,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the plan I selected.,Thanks for reaching out about the billing issue. We will look into this and get back to you within the next 48 hours.,Adjusted the invoice and issued a refund where applicable.,61.32,0,very_negative,2,web,other,2023-08-27 16:08:33
3,2024-09-24T14:04:53,education,email,mobile_app,performance,low,closed_no_action,gold,Queries in the mobile app module are timing out.,We understand performance issues can be frustrating. We will look into this and get back to you within the next 48 hours.,Ticket closed without further action after no response from the customer.,226.93,0,positive,5,ios,other,2024-09-24 14:04:53
4,2024-11-01T16:14:51,education,in_app,billing,security_concern,urgent,resolved,gold,I noticed a suspicious login on my account.,We take security very seriously. This has been escalated to our on-call team for immediate attention.,"Investigated logs, mitigated the issue, and updated security settings.",4.36,0,negative,3,api_client,APAC,2024-11-01 16:14:51


In [111]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [112]:
print (df['issue_type'].nunique())
print (df['issue_type'].value_counts())


8
issue_type
how_to              7711
account_access      7560
performance         7524
other               7488
feature_request     7480
billing_problem     7463
security_concern    7459
bug                 7428
Name: count, dtype: int64


In [113]:
print (df['initial_message'].nunique())
print (df['initial_message'].value_counts())

96
initial_message
I downgraded my plan but I am still being billed for the higher tier.              2602
My 2FA code is not working when I try to sign in.                                  2560
Something seems off, can you please check my workspace?                            2558
My account was locked after multiple failed login attempts.                        2536
I have a general question about my account.                                        2513
We need details about your data encryption and compliance.                         2497
I noticed a suspicious login on my account.                                        2481
Overall performance has degraded in the last few days.                             2476
I cannot log in; the system says my password is incorrect.                         2464
I was charged twice for my subscription this month.                                2434
My invoice amount is incorrect compared to the plan I selected.                    2427
I need help b

In [114]:
df_copy = df.copy()

## first solution

In [116]:
from sklearn.utils import resample

df_small = df.drop_duplicates(subset=['initial_message'])

# df_expanded = pd.concat([df_small]*20, ignore_index=True)

In [117]:
# df_expanded.shape

In [118]:
df.shape

(60113, 18)

## second solution

In [119]:
df_small.shape

(96, 18)

In [120]:
df1_small = df_small.copy()

In [121]:
def refine_issue(row):
    text = row['initial_message'].lower()
    issue = row['issue_type']

    # =========================
    # 1. ACCOUNT ACCESS
    # =========================
    if issue == 'account_access':
        if 'password' in text:
            return 'account_password_issue'
        elif 'locked' in text:
            return 'account_locked'
        elif '2fa' in text or 'code' in text:
            return 'account_2fa_issue'
        elif 'log in' in text or 'login' in text:
            return 'account_login_failure'
        else:
            return 'account_other'

    # =========================
    # 2. BILLING
    # =========================
    elif issue == 'billing_problem':
        if 'charged twice' in text or 'double' in text:
            return 'billing_double_charge'
        elif 'invoice' in text:
            return 'billing_invoice_issue'
        elif 'plan' in text:
            return 'billing_plan_mismatch'
        elif 'refund' in text:
            return 'billing_refund_request'
        else:
            return 'billing_other'

    # =========================
    # 3. PERFORMANCE
    # =========================
    elif issue == 'performance':
        if 'slow' in text:
            return 'performance_slow'
        elif 'timing out' in text or 'timeout' in text:
            return 'performance_timeout'
        elif 'crashing' in text or 'crash' in text:
            return 'performance_crash'
        elif 'degraded' in text:
            return 'performance_degradation'
        else:
            return 'performance_other'

    # =========================
    # 4. BUG
    # =========================
    elif issue == 'bug':
        if 'error message' in text:
            return 'bug_error_message'
        elif 'not saving' in text:
            return 'bug_data_not_saved'
        elif 'crashing' in text:
            return 'bug_crash'
        else:
            return 'bug_other'

    # =========================
    # 5. FEATURE REQUEST
    # =========================
    elif issue == 'feature_request':
        if 'add an option' in text or 'customize' in text:
            return 'feature_customization'
        elif 'would be great' in text or 'support' in text:
            return 'feature_new_feature'
        elif 'advanced options' in text:
            return 'feature_advanced_options'
        else:
            return 'feature_other'

    # =========================
    # 6. HOW TO
    # =========================
    elif issue == 'how_to':
        if 'how can i' in text or 'how do i' in text:
            return 'how_to_usage'
        elif 'set up' in text or 'setup' in text:
            return 'how_to_setup'
        elif 'configure' in text:
            return 'how_to_configuration'
        elif 'export' in text:
            return 'how_to_export'
        else:
            return 'how_to_other'

    # =========================
    # 7. SECURITY
    # =========================
    elif issue == 'security_concern':
        if 'suspicious' in text:
            return 'security_suspicious_activity'
        elif 'vulnerability' in text:
            return 'security_vulnerability'
        elif 'encryption' in text or 'compliance' in text:
            return 'security_compliance'
        else:
            return 'security_other'

    # =========================
    # 8. OTHER (we will REMOVE later)
    # =========================
    elif issue == 'other':
        if 'not sure' in text:
            return 'other_unclear'
        elif 'general question' in text:
            return 'other_general'
        else:
            return 'other_misc'

    return issue

In [122]:
df1_small['refined_issue'] = df.apply(refine_issue, axis=1)

In [123]:
df1_small.shape

(96, 19)

In [124]:
df1_small[['issue_type','refined_issue','initial_message']].head(10)

,issue_type,refined_issue,initial_message
0,account_access,account_password_issue,I cannot log in; the system says my password is incorrect.
1,security_concern,security_suspicious_activity,I noticed a suspicious login on my account.
2,billing_problem,billing_invoice_issue,My invoice amount is incorrect compared to the plan I selected.
3,performance,performance_timeout,Queries in the mobile app module are timing out.
5,billing_problem,billing_plan_mismatch,I downgraded my plan but I am still being billed for the higher tier.
6,billing_problem,billing_double_charge,I was charged twice for my subscription this month.
7,performance,performance_degradation,Overall performance has degraded in the last few days.
9,performance,performance_slow,The data export page is very slow and takes a long time to load.
10,bug,bug_crash,The page keeps crashing whenever I try to open the billing section.
11,security_concern,security_compliance,We need details about your data encryption and compliance.


In [125]:
df1_small['refined_issue'].value_counts()

refined_issue
feature_customization           7
feature_new_feature             7
performance_timeout             7
security_vulnerability          7
performance_slow                7
bug_crash                       7
bug_error_message               7
bug_data_not_saved              7
feature_advanced_options        7
how_to_usage                    7
how_to_configuration            7
how_to_other                    6
account_2fa_issue               1
other_misc                      1
other_unclear                   1
account_password_issue          1
other_general                   1
account_locked                  1
security_suspicious_activity    1
security_compliance             1
performance_degradation         1
billing_double_charge           1
billing_plan_mismatch           1
billing_invoice_issue           1
how_to_export                   1
Name: count, dtype: int64

In [126]:
df1_small['refined_issue'].nunique()

25

## another version

In [127]:
df2_small = df_small.copy()

In [128]:
def refine_issue_2(row):
    text = row['initial_message'].lower()
    issue = row['issue_type']

    # ACCOUNT
    if issue == 'account_access':
        if 'password' in text:
            return 'account_password_issue'
        elif 'locked' in text:
            return 'account_locked'
        elif '2fa' in text or 'code' in text:
            return 'account_2fa_issue'
        else:
            return 'account_login_failure'

    # BILLING
    elif issue == 'billing_problem':
        if 'invoice' in text:
            return 'billing_invoice_issue'
        elif 'plan' in text or 'subscription' in text:
            return 'billing_subscription_issue'
        else:
            return 'billing_payment_issue'

    # PERFORMANCE
    elif issue == 'performance':
        if 'slow' in text:
            return 'performance_slow'
        elif 'timeout' in text or 'timing out' in text:
            return 'performance_timeout'
        else:
            return 'performance_crash'

    # BUG
    elif issue == 'bug':
        if 'error' in text:
            return 'bug_error'
        elif 'not saving' in text:
            return 'bug_data_loss'
        else:
            return 'bug_ui_issue'

    # FEATURE REQUEST
    elif issue == 'feature_request':
        if 'advanced' in text or 'customize' in text:
            return 'feature_request_improvement'
        else:
            return 'feature_request_new'

    # HOW TO
    elif issue == 'how_to':
        if 'set up' in text:
            return 'how_to_setup'
        elif 'configure' in text:
            return 'how_to_configuration'
        else:
            return 'how_to_usage'

    # SECURITY
    elif issue == 'security_concern':
        if 'vulnerability' in text:
            return 'security_vulnerability'
        elif 'compliance' in text or 'encryption' in text:
            return 'security_compliance'
        else:
            return 'security_alert'

    # OTHER
    else:
        return 'general_inquiry'

In [129]:
df2_small['refined_issue'] = df.apply(refine_issue_2, axis=1)


In [130]:
df2_small.shape

(96, 19)

In [131]:
df2_small[['issue_type','refined_issue','initial_message']].head(10)

,issue_type,refined_issue,initial_message
0,account_access,account_password_issue,I cannot log in; the system says my password is incorrect.
1,security_concern,security_alert,I noticed a suspicious login on my account.
2,billing_problem,billing_invoice_issue,My invoice amount is incorrect compared to the plan I selected.
3,performance,performance_timeout,Queries in the mobile app module are timing out.
5,billing_problem,billing_subscription_issue,I downgraded my plan but I am still being billed for the higher tier.
6,billing_problem,billing_subscription_issue,I was charged twice for my subscription this month.
7,performance,performance_crash,Overall performance has degraded in the last few days.
9,performance,performance_slow,The data export page is very slow and takes a long time to load.
10,bug,bug_ui_issue,The page keeps crashing whenever I try to open the billing section.
11,security_concern,security_compliance,We need details about your data encryption and compliance.


In [132]:
df2_small['refined_issue'].value_counts()

refined_issue
how_to_usage                   14
feature_request_improvement    14
bug_data_loss                   7
bug_ui_issue                    7
security_vulnerability          7
bug_error                       7
how_to_configuration            7
feature_request_new             7
performance_slow                7
performance_timeout             7
general_inquiry                 3
billing_subscription_issue      2
security_compliance             1
security_alert                  1
performance_crash               1
account_locked                  1
account_2fa_issue               1
billing_invoice_issue           1
account_password_issue          1
Name: count, dtype: int64

In [133]:
df2_small['initial_message'].value_counts()

initial_message
I cannot log in; the system says my password is incorrect.                         1
I noticed a suspicious login on my account.                                        1
Our team needs more advanced options for analytics dashboard.                      1
Our team needs more advanced options for data export.                              1
I am seeing an error message when I click on login auth.                           1
The page keeps crashing whenever I try to open the api integration section.        1
The api integration page is very slow and takes a long time to load.               1
I am seeing an error message when I click on billing.                              1
Queries in the login auth module are timing out.                                   1
Our team needs more advanced options for billing.                                  1
It would be great to have api integration support in your product.                 1
I am seeing an error message when I click on mobi

In [134]:
df2_small.info()

<class 'pandas.core.frame.DataFrame'>
Index: 96 entries, 0 to 661
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   created_at             96 non-null     object 
 1   customer_segment       96 non-null     object 
 2   channel                96 non-null     object 
 3   product_area           96 non-null     object 
 4   issue_type             96 non-null     object 
 5   priority               96 non-null     object 
 6   status                 96 non-null     object 
 7   sla_plan               96 non-null     object 
 8   initial_message        96 non-null     object 
 9   agent_first_reply      96 non-null     object 
 10  resolution_summary     96 non-null     object 
 11  resolution_time_hours  96 non-null     float64
 12  reopened               96 non-null     int64  
 13  customer_sentiment     96 non-null     object 
 14  csat_score             96 non-null     int64  
 15  platform    

In [135]:
import random

def paraphrase_advanced(text):
    text = text.lower()

    templates = [
        "I'm experiencing an issue where {}",
        "There seems to be a problem: {}",
        "I'm having trouble because {}",
        "Something is wrong — {}",
        "Can you help? {}",
        "I'm facing a situation where {}",
        "It appears that {}",
        "I noticed that {}",
        "Currently dealing with this issue: {}",
        "This keeps happening: {}"
    ]

    synonyms = {
        "log in": ["sign in", "access my account", "log into my account"],
        "error message": ["an error appears", "I get an error", "an error pops up"],
        "slow": ["very slow", "lagging", "taking too long"],
        "crashing": ["closing unexpectedly", "crashes", "stops working"],
        "not saving": ["fails to save", "doesn't save", "won't save changes"],
        "timing out": ["times out", "keeps timing out", "request times out"],
        "charged twice": ["billed twice", "double charged", "charged two times"],
        "invoice": ["billing statement", "invoice details"],
        "password": ["login password", "account password"],
    }

    results = []

    for _ in range(5):  # generate multiple variations
        new_text = text

        for key, vals in synonyms.items():
            if key in new_text:
                new_text = new_text.replace(key, random.choice(vals))

        template = random.choice(templates)
        new_text = template.format(new_text)

        # small noise
        if random.random() > 0.5:
            new_text += " " + random.choice([
                "since yesterday",
                "after the last update",
                "for a while now",
                "every time I try"
            ])

        results.append(new_text)

    return results

In [136]:
TARGET = 100

In [137]:
import pandas as pd

balanced_rows = []

for category in df2_small['refined_issue'].unique():
    subset = df2_small[df2_small['refined_issue'] == category]

    current_size = len(subset)

    if current_size == 0:
        continue

    rows = subset.to_dict('records')

    # Keep original
    balanced_rows.extend(rows)

    # Generate more until TARGET
    while len([r for r in balanced_rows if r['refined_issue'] == category]) < TARGET:
        for row in rows:
            variations = paraphrase_advanced(row['initial_message'])

            for v in variations:
                new_row = row.copy()
                new_row['initial_message'] = v
                new_row['issue_type'] = None
                balanced_rows.append(new_row)

                if len([r for r in balanced_rows if r['refined_issue'] == category]) >= TARGET:
                    break
            if len([r for r in balanced_rows if r['refined_issue'] == category]) >= TARGET:
                break

df2_small_balanced = pd.DataFrame(balanced_rows)

In [138]:
print(df2_small_balanced['initial_message'].nunique())
print(df2_small_balanced['initial_message'].value_counts())

1394
initial_message
I noticed that my 2fa code is not working when i try to sign in.                                                                               10
Currently dealing with this issue: my account was locked after multiple failed login attempts.                                                  9
This keeps happening: i noticed a suspicious login on my account.                                                                               9
I'm facing a situation where we need details about your data encryption and compliance.                                                         8
I'm experiencing an issue where i noticed a suspicious login on my account.                                                                     8
Currently dealing with this issue: i noticed a suspicious login on my account.                                                                  8
There seems to be a problem: overall performance has degraded in the last few days.                    

In [139]:
df2_small_balanced.shape

(1900, 19)

In [140]:
category = df2_small_balanced.loc[
    df2_small_balanced['initial_message'] == 'There seems to be a problem: my account was locked after multiple failed login attempts.',
    'refined_issue'  
]

print(category)

1217    account_locked
1229    account_locked
Name: refined_issue, dtype: object


In [141]:
print(df2_small_balanced['refined_issue'].nunique())
print(df2_small_balanced['initial_message'].nunique())

19
1394


In [142]:
print(df2_small_balanced['refined_issue'].unique())
print(df2_small_balanced['refined_issue'].value_counts())

['account_password_issue' 'security_alert' 'billing_invoice_issue'
 'performance_timeout' 'billing_subscription_issue' 'performance_crash'
 'performance_slow' 'bug_ui_issue' 'security_compliance' 'bug_data_loss'
 'feature_request_improvement' 'how_to_configuration' 'account_locked'
 'general_inquiry' 'account_2fa_issue' 'how_to_usage' 'bug_error'
 'security_vulnerability' 'feature_request_new']
refined_issue
account_password_issue         100
feature_request_improvement    100
security_vulnerability         100
bug_error                      100
how_to_usage                   100
account_2fa_issue              100
general_inquiry                100
account_locked                 100
how_to_configuration           100
bug_data_loss                  100
security_alert                 100
security_compliance            100
bug_ui_issue                   100
performance_slow               100
performance_crash              100
billing_subscription_issue     100
performance_timeout         

In [143]:
df2_small_balanced.head()

,created_at,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,platform,region,date,refined_issue
0,2024-01-31T05:14:27,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27,account_password_issue
1,2024-01-31T05:14:27,individual,email,data_export,None,low,resolved,standard,Currently dealing with this issue: i cannot access my account; the system says my account password is incorrect. after the last update,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27,account_password_issue
2,2024-01-31T05:14:27,individual,email,data_export,None,low,resolved,standard,It appears that i cannot sign in; the system says my account password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27,account_password_issue
3,2024-01-31T05:14:27,individual,email,data_export,None,low,resolved,standard,It appears that i cannot access my account; the system says my login password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27,account_password_issue
4,2024-01-31T05:14:27,individual,email,data_export,None,low,resolved,standard,It appears that i cannot sign in; the system says my account password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27,account_password_issue


In [144]:
df2_small_balanced.duplicated().sum()

506

In [145]:
df2_small_balanced.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1900 entries, 0 to 1899
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   created_at             1900 non-null   object 
 1   customer_segment       1900 non-null   object 
 2   channel                1900 non-null   object 
 3   product_area           1900 non-null   object 
 4   issue_type             96 non-null     object 
 5   priority               1900 non-null   object 
 6   status                 1900 non-null   object 
 7   sla_plan               1900 non-null   object 
 8   initial_message        1900 non-null   object 
 9   agent_first_reply      1900 non-null   object 
 10  resolution_summary     1900 non-null   object 
 11  resolution_time_hours  1900 non-null   float64
 12  reopened               1900 non-null   int64  
 13  customer_sentiment     1900 non-null   object 
 14  csat_score             1900 non-null   int64  
 15  plat

In [146]:
for col in df2_small_balanced.columns:
    if col not in ['initial_message', 'refined_issue', 'issue_type','created_at','resolution_time_hours','date','agent_first_reply','resolution_summary']:
        unique_vals = df2_small_balanced[col].nunique()
        unique_vals_list = df2_small_balanced[col].unique()
        print(f"Column '{col}' has {unique_vals} unique values = {unique_vals_list}")

# Column 'agent_first_reply' has 25 unique values

Column 'customer_segment' has 5 unique values = ['individual' 'enterprise' 'education' 'non_profit' 'small_business']
Column 'channel' has 5 unique values = ['email' 'in_app' 'phone_transcript' 'web_form' 'chat']
Column 'product_area' has 7 unique values = ['data_export' 'billing' 'login_auth' 'mobile_app' 'analytics_dashboard'
 'api_integration' 'notifications']
Column 'priority' has 4 unique values = ['low' 'medium' 'high' 'urgent']
Column 'status' has 2 unique values = ['resolved' 'closed_no_action']
Column 'sla_plan' has 3 unique values = ['standard' 'gold' 'platinum']
Column 'reopened' has 2 unique values = [0 1]
Column 'customer_sentiment' has 5 unique values = ['very_negative' 'neutral' 'positive' 'negative' 'very_positive']
Column 'csat_score' has 6 unique values = [1 3 2 5 4 0]
Column 'platform' has 5 unique values = ['android' 'web' 'ios' 'desktop_app' 'api_client']
Column 'region' has 5 unique values = ['EU' 'other' 'LATAM' 'APAC' 'MEA']


## now i will try to make it 10k

In [98]:
df_temp = df2_small_balanced.copy()

In [99]:
df_temp.shape

(1900, 19)

In [100]:
import pandas as pd
import random
import numpy as np
from datetime import datetime, timedelta
  # or however you load your data
target_rows = 10000

# Unique values for randomization
customer_segments = ['individual', 'enterprise', 'education', 'non_profit', 'small_business']
channels = ['email', 'in_app', 'phone_transcript', 'web_form', 'chat']
product_areas = ['data_export', 'billing', 'login_auth', 'mobile_app', 'analytics_dashboard', 'api_integration', 'notifications']
priorities = ['low', 'medium', 'high', 'urgent']
statuses = ['resolved', 'closed_no_action']
sla_plans = ['standard', 'gold', 'platinum']
reopened_vals = [0, 1]
customer_sentiments = ['very_negative', 'neutral', 'positive', 'negative', 'very_positive']
csat_scores = [0, 1, 2, 3, 4, 5]
platforms = ['android', 'web', 'ios', 'desktop_app', 'api_client']
regions = ['EU', 'other', 'LATAM', 'APAC', 'MEA']

all_rows = []

categories = df_temp['refined_issue'].unique()
rows_per_category = target_rows // len(categories)
remaining_rows = target_rows - (rows_per_category * len(categories))

for i, cat in enumerate(categories):
    messages = df_temp[df_temp['refined_issue'] == cat]['initial_message'].tolist()
    cat_rows = rows_per_category + (1 if i < remaining_rows else 0)
    
    for _ in range(cat_rows):
        base_msg = random.choice(messages)
        
        # More comprehensive text variations
        variations = {
            "help": random.choice(["assist", "support", "aid", "help"]),
            "issue": random.choice(["problem", "error", "bug", "glitch", "issue"]),
            "page": random.choice(["screen", "window", "interface", "page", "panel"]),
            "can't": random.choice(["cannot", "unable to", "can't"]),
            "not working": random.choice(["not functioning", "broken", "down", "not working"])
        }
        
        new_msg = base_msg
        for old, new in variations.items():
            new_msg = new_msg.replace(old, new)
        
        # Generate realistic timestamps
        created_date = datetime.now() - timedelta(days=random.randint(0, 365), 
                                                   hours=random.randint(0, 23), 
                                                   minutes=random.randint(0, 59))
        
        # Realistic resolution times based on priority
        priority = random.choice(priorities)
        if priority == 'urgent':
            res_time = round(random.uniform(0.5, 4), 2)
        elif priority == 'high':
            res_time = round(random.uniform(2, 24), 2)
        elif priority == 'medium':
            res_time = round(random.uniform(12, 72), 2)
        else:
            res_time = round(random.uniform(48, 200), 2)
        
        # Generate correlated sentiment and CSAT score
        sentiment = random.choice(customer_sentiments)
        if sentiment == 'very_positive':
            csat = random.choice([4, 5, 5])
        elif sentiment == 'positive':
            csat = random.choice([3, 4, 4])
        elif sentiment == 'neutral':
            csat = random.choice([2, 3])
        elif sentiment == 'negative':
            csat = random.choice([1, 2])
        else:  # very_negative
            csat = random.choice([0, 1])
        
        row = {
            'created_at': created_date,
            'customer_segment': random.choice(customer_segments),
            'channel': random.choice(channels),
            'product_area': random.choice(product_areas),
            'priority': priority,
            'status': random.choice(statuses),
            'sla_plan': random.choice(sla_plans),
            'initial_message': new_msg,
            'agent_first_reply': created_date + timedelta(minutes=random.randint(5, 180)),
            'resolution_summary': f"Resolved {random.choice(['quickly', 'after investigation', 'with workaround'])}",
            'resolution_time_hours': res_time,
            'reopened': 1 if random.random() < 0.15 else 0,  # 15% reopen rate
            'customer_sentiment': sentiment,
            'csat_score': csat,
            'platform': random.choice(platforms),
            'region': random.choice(regions),
            'refined_issue': cat
        }
        all_rows.append(row)

df_expanded = pd.DataFrame(all_rows)
print("Expanded dataset shape:", df_expanded.shape)
print(df_expanded['refined_issue'].value_counts())
print(f"Total rows: {len(df_expanded)}")

Expanded dataset shape: (10000, 17)
refined_issue
account_password_issue         527
billing_invoice_issue          527
performance_timeout            527
billing_subscription_issue     527
performance_crash              527
security_alert                 527
account_locked                 526
security_vulnerability         526
bug_error                      526
how_to_usage                   526
account_2fa_issue              526
general_inquiry                526
bug_data_loss                  526
how_to_configuration           526
feature_request_improvement    526
security_compliance            526
bug_ui_issue                   526
performance_slow               526
feature_request_new            526
Name: count, dtype: int64
Total rows: 10000


In [102]:
 df_expanded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   created_at             10000 non-null  datetime64[ns]
 1   customer_segment       10000 non-null  object        
 2   channel                10000 non-null  object        
 3   product_area           10000 non-null  object        
 4   priority               10000 non-null  object        
 5   status                 10000 non-null  object        
 6   sla_plan               10000 non-null  object        
 7   initial_message        10000 non-null  object        
 8   agent_first_reply      10000 non-null  datetime64[ns]
 9   resolution_summary     10000 non-null  object        
 10  resolution_time_hours  10000 non-null  float64       
 11  reopened               10000 non-null  int64         
 12  customer_sentiment     10000 non-null  object        
 13  cs

In [103]:
 df_expanded.sample(5)

,created_at,customer_segment,channel,product_area,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,platform,region,refined_issue
2737,2025-06-20 04:58:04.613299,education,email,billing,low,closed_no_action,gold,Currently dealing with this glitch: overall performance has degraded in the last few days.,2025-06-20 07:54:04.613299,Resolved with workaround,90.68,0,neutral,2,ios,APAC,performance_crash
9379,2025-06-03 03:02:05.047341,education,web_form,api_integration,medium,resolved,gold,I noticed that there seems to be a vulnerability in the mobile app functionality.,2025-06-03 03:29:05.047341,Resolved with workaround,44.89,0,positive,4,ios,MEA,security_vulnerability
2364,2025-07-31 08:07:04.589080,education,web_form,login_auth,medium,resolved,standard,There seems to be a problem: i was billed twice for my subscription this month.,2025-07-31 08:44:04.589080,Resolved after investigation,38.52,0,very_positive,5,api_client,MEA,billing_subscription_issue
6785,2025-08-22 12:59:04.866311,small_business,web_form,analytics_dashboard,urgent,closed_no_action,standard,I'm having trouble because my account was locked after multiple failed login attempts. after the last update,2025-08-22 14:33:04.866311,Resolved quickly,3.35,0,very_negative,1,android,EU,account_locked
6424,2025-04-14 10:29:04.850273,enterprise,email,notifications,medium,resolved,platinum,I'm facing a situation where my account was locked after multiple failed login attempts. for a while now,2025-04-14 12:59:04.850273,Resolved with workaround,49.46,0,very_negative,1,web,APAC,account_locked


In [104]:
 df_expanded.head()

,created_at,customer_segment,channel,product_area,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,platform,region,refined_issue
0,2025-12-21 00:53:04.444492,enterprise,email,api_integration,medium,resolved,platinum,I'm facing a situation where i cannot log into my account; the system says my login password is incorrect.,2025-12-21 02:11:04.444492,Resolved after investigation,58.42,0,negative,1,api_client,LATAM,account_password_issue
1,2025-12-10 06:57:04.444492,individual,chat,data_export,high,resolved,platinum,Currently dealing with this problem: i cannot access my account; the system says my login password is incorrect.,2025-12-10 07:41:04.444492,Resolved with workaround,21.75,0,positive,4,android,EU,account_password_issue
2,2025-12-17 02:24:04.445517,enterprise,chat,api_integration,urgent,closed_no_action,gold,I'm having trouble because i cannot log into my account; the system says my login password is incorrect. for a while now,2025-12-17 03:20:04.445517,Resolved quickly,1.71,0,very_positive,4,web,other,account_password_issue
3,2025-11-26 09:33:04.445517,enterprise,web_form,mobile_app,medium,closed_no_action,standard,I noticed that i cannot sign in; the system says my account password is incorrect.,2025-11-26 09:41:04.445517,Resolved with workaround,28.27,0,negative,2,ios,MEA,account_password_issue
4,2025-08-27 00:51:04.445671,individual,chat,api_integration,low,resolved,platinum,I'm having trouble because i cannot sign in; the system says my login password is incorrect.,2025-08-27 01:59:04.445671,Resolved after investigation,115.03,0,positive,3,android,LATAM,account_password_issue


In [105]:
df2_small_balanced.head()

,created_at,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,platform,region,date,refined_issue
0,2024-01-31T05:14:27,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27,account_password_issue
1,2024-01-31T05:14:27,individual,email,data_export,None,low,resolved,standard,Something is wrong — i cannot sign in; the system says my account password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27,account_password_issue
2,2024-01-31T05:14:27,individual,email,data_export,None,low,resolved,standard,I'm facing a situation where i cannot sign in; the system says my login password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27,account_password_issue
3,2024-01-31T05:14:27,individual,email,data_export,None,low,resolved,standard,I'm having trouble because i cannot access my account; the system says my account password is incorrect. every time I try,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27,account_password_issue
4,2024-01-31T05:14:27,individual,email,data_export,None,low,resolved,standard,I'm facing a situation where i cannot log into my account; the system says my login password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27,account_password_issue


In [ ]:
# save it

## another v


In [147]:
df_temp2 = df2_small_balanced.copy()

In [148]:
# import pandas as pd
# import random
# from datetime import datetime, timedelta

# target_rows = 10000

# customer_segments = ['individual', 'enterprise', 'education', 'non_profit', 'small_business']
# channels = ['email', 'in_app', 'phone_transcript', 'web_form', 'chat']
# product_areas = ['data_export', 'billing', 'login_auth', 'mobile_app', 'analytics_dashboard', 'api_integration', 'notifications']
# priorities = ['low', 'medium', 'high', 'urgent']
# statuses = ['resolved', 'closed_no_action']
# sla_plans = ['standard', 'gold', 'platinum']
# reopened_vals = [0, 1]
# customer_sentiments = ['very_negative', 'neutral', 'positive', 'negative', 'very_positive']
# csat_scores = [0, 1, 2, 3, 4, 5]
# platforms = ['android', 'web', 'ios', 'desktop_app', 'api_client']
# regions = ['EU', 'other', 'LATAM', 'APAC', 'MEA']

# all_rows = []

# categories = df['refined_issue'].unique()
# rows_per_category = target_rows // len(categories)

# for cat in categories:
#     messages = df[df['refined_issue'] == cat]['initial_message'].tolist()
    
#     for _ in range(rows_per_category):
#         base_msg = random.choice(messages)
        
#         new_msg = base_msg.replace("help", random.choice(["assist", "support", "aid"]))
#         new_msg = new_msg.replace("issue", random.choice(["problem", "error", "bug"]))
#         new_msg = new_msg.replace("page", random.choice(["screen", "window", "interface"]))
        
#         row = {
#             'created_at': datetime.now() - timedelta(days=random.randint(0, 365)),
#             'customer_segment': random.choice(customer_segments),
#             'channel': random.choice(channels),
#             'product_area': random.choice(product_areas),
#             'issue_type': None,
#             'priority': random.choice(priorities),
#             'status': random.choice(statuses),
#             'sla_plan': random.choice(sla_plans),
#             'initial_message': new_msg,
#             'agent_first_reply': '',
#             'resolution_summary': '',
#             'resolution_time_hours': round(random.uniform(1, 200), 2),
#             'reopened': random.choice(reopened_vals),
#             'customer_sentiment': random.choice(customer_sentiments),
#             'csat_score': random.choice(csat_scores),
#             'platform': random.choice(platforms),
#             'region': random.choice(regions),
#             'date': datetime.now() - timedelta(days=random.randint(0, 365)),
#             'refined_issue': cat
#         }
#         all_rows.append(row)

# df_expanded_2 = pd.DataFrame(all_rows)
# print("Expanded dataset shape:", df_expanded_2.shape)
# print(df_expanded_2['refined_issue'].value_counts())

In [149]:
df_temp2.shape

(1900, 19)

In [150]:
df_temp2['initial_message'].unique()

array(['I cannot log in; the system says my password is incorrect.',
       'Currently dealing with this issue: i cannot access my account; the system says my account password is incorrect. after the last update',
       'It appears that i cannot sign in; the system says my account password is incorrect.',
       ...,
       'Can you help? it would be great to have api integration support in your product. after the last update',
       'It appears that it would be great to have analytics dashboard support in your product. for a while now',
       'I noticed that it would be great to have analytics dashboard support in your product.'],
      dtype=object)

In [151]:
df_temp2['refined_issue'].unique()

array(['account_password_issue', 'security_alert',
       'billing_invoice_issue', 'performance_timeout',
       'billing_subscription_issue', 'performance_crash',
       'performance_slow', 'bug_ui_issue', 'security_compliance',
       'bug_data_loss', 'feature_request_improvement',
       'how_to_configuration', 'account_locked', 'general_inquiry',
       'account_2fa_issue', 'how_to_usage', 'bug_error',
       'security_vulnerability', 'feature_request_new'], dtype=object)

In [175]:
# Get unique messages for each category separately
for cat in ['account_2fa_issue', 'how_to_usage', 'bug_error']:
    cat_messages = df2_small_balanced[df2_small_balanced['refined_issue'] == cat]['initial_message'].unique()
    print(f"\nCategory: {cat}")
    print(f"Number of unique messages: {len(cat_messages)}")
    print("Sample messages:")
    for i, msg in enumerate(cat_messages[:]):
        print(f"  {i+1}. {msg}")


Category: account_2fa_issue
Number of unique messages: 36
Sample messages:
  1. My 2FA code is not working when I try to sign in.
  2. It appears that my 2fa code is not working when i try to sign in. for a while now
  3. Can you help? my 2fa code is not working when i try to sign in.
  4. It appears that my 2fa code is not working when i try to sign in.
  5. I'm having trouble because my 2fa code is not working when i try to sign in. for a while now
  6. This keeps happening: my 2fa code is not working when i try to sign in. for a while now
  7. Something is wrong — my 2fa code is not working when i try to sign in. every time I try
  8. Currently dealing with this issue: my 2fa code is not working when i try to sign in.
  9. There seems to be a problem: my 2fa code is not working when i try to sign in.
  10. This keeps happening: my 2fa code is not working when i try to sign in.
  11. There seems to be a problem: my 2fa code is not working when i try to sign in. after the last update

In [155]:
import random

login_actions = [
    "log in", "sign in", "access my account", "enter my account"
]

cannot_phrases = [
    "I can't", "I cannot", "I'm unable to", "I am not able to"
]

password_errors = [
    "the password is incorrect",
    "my password is wrong",
    "invalid password",
    "password not accepted",
    "incorrect credentials"
]

prefixes = [
    "",
    "I'm having trouble, ",
    "I’m facing an issue where ",
    "There is a problem, ",
    "Something is wrong, "
]

suffixes = [
    "",
    " since yesterday",
    " for a while now",
    " every time I try",
    " after the last update"
]

In [156]:
def paraphrase_account_issue(text, n=3):
    results = set()

    for _ in range(n * 3):
        new_text = text.lower()

        # Replace login actions
        if "log" in new_text or "sign" in new_text or "access" in new_text:
            new_text = random.choice(cannot_phrases) + " " + random.choice(login_actions)

        # Replace password part
        if "password" in text:
            new_text += "; " + random.choice(password_errors)

        # Add prefix/suffix
        new_text = random.choice(prefixes) + new_text + random.choice(suffixes)

        results.add(new_text)

        if len(results) >= n:
            break

    return list(results)

In [157]:
security_actions = [
    "I noticed a suspicious login",
    "There was an unusual login",
    "I detected an unknown login",
    "Someone accessed my account",
    "I saw a login I don't recognize"
]

def paraphrase_security(text, n=3):
    results = set()

    for _ in range(n * 3):
        new_text = random.choice(security_actions)
        new_text += random.choice([
            "",
            " on my account",
            " from an unknown device",
            " from a different location"
        ])

        new_text = random.choice(prefixes) + new_text + random.choice(suffixes)

        results.add(new_text)

        if len(results) >= n:
            break

    return list(results)

In [158]:
augmented_rows = []

for _, row in df_temp2.iterrows():
    text = row['initial_message']
    label = row['refined_issue']

    if label == "account_password_issue":
        new_texts = paraphrase_account_issue(text, n=3)

    elif label == "security_alert":
        new_texts = paraphrase_security(text, n=3)

    else:
        continue

    for t in new_texts:
        augmented_rows.append({
            "initial_message": t,
            "refined_issue": label
        })

df_aug = pd.DataFrame(augmented_rows)

df_final = pd.concat([df_temp2, df_aug])
df_final.drop_duplicates(subset=['initial_message'], inplace=True)

In [159]:
df_final.shape

(1899, 19)

In [161]:
df_final['refined_issue'].value_counts()

refined_issue
account_password_issue         364
security_alert                 259
performance_timeout             97
feature_request_improvement     96
bug_ui_issue                    95
bug_data_loss                   95
how_to_usage                    92
bug_error                       92
performance_slow                91
feature_request_new             86
how_to_configuration            85
security_vulnerability          85
billing_subscription_issue      70
general_inquiry                 70
billing_invoice_issue           61
account_locked                  44
performance_crash               41
security_compliance             40
account_2fa_issue               36
Name: count, dtype: int64

In [165]:
billing_objects = [
    "invoice", "billing statement", "bill", "payment summary", "invoice details"
]

amount_issues = [
    "amount is incorrect",
    "charged amount is wrong",
    "total is not correct",
    "billing amount doesn't match",
    "incorrect charge on my account"
]

plan_reference = [
    "compared to my plan",
    "compared to the plan I selected",
    "based on my subscription",
    "according to my plan",
    "than expected for my plan"
]

def paraphrase_billing_issue(text, n=3):
    results = set()

    for _ in range(n * 3):
        obj = random.choice(billing_objects)
        issue = random.choice(amount_issues)
        plan = random.choice(plan_reference)

        new_text = f"My {obj} {issue} {plan}"
        new_text = random.choice(prefixes) + new_text + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

In [166]:
modules = [
    "mobile app",
    "analytics dashboard",
    "data export",
    "api integration",
    "login system",
    "notifications",
    "billing system"
]

timeout_phrases = [
    "is timing out",
    "keeps timing out",
    "request times out",
    "fails due to timeout",
    "is too slow and times out"
]

query_terms = [
    "queries",
    "requests",
    "operations",
    "calls",
    "processes"
]

def paraphrase_timeout_issue(text, n=3):
    results = set()

    for _ in range(n * 3):
        module = random.choice(modules)
        timeout = random.choice(timeout_phrases)
        query = random.choice(query_terms)

        new_text = f"{query} in the {module} module {timeout}"
        new_text = random.choice(prefixes) + new_text + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

In [168]:
augmented_rows = []

for _, row in df_temp2.iterrows():
    text = row['initial_message']
    label = row['refined_issue']


    if label == "billing_invoice_issue":
        new_texts = paraphrase_billing_issue(text, n=3)

    elif label == "performance_timeout":
        new_texts = paraphrase_timeout_issue(text, n=3)

    else:
        continue

    for t in new_texts:
        augmented_rows.append({
            "initial_message": t,
            "refined_issue": label
        })

df_aug = pd.DataFrame(augmented_rows)
df_final = pd.concat([df_final, df_aug])
df_final.drop_duplicates(subset=['initial_message'], inplace=True)

In [169]:
df_final.shape

(2475, 19)

In [170]:
df_final['refined_issue'].value_counts()

refined_issue
performance_timeout            386
account_password_issue         364
billing_invoice_issue          348
security_alert                 259
feature_request_improvement     96
bug_ui_issue                    95
bug_data_loss                   95
how_to_usage                    92
bug_error                       92
performance_slow                91
feature_request_new             86
how_to_configuration            85
security_vulnerability          85
billing_subscription_issue      70
general_inquiry                 70
account_locked                  44
performance_crash               41
security_compliance             40
account_2fa_issue               36
Name: count, dtype: int64

In [176]:
subscription_actions = [
    "i downgraded my plan but i am still being billed for the higher tier",
    "i switched to a lower plan but charges are still for the higher one",
    "i changed my subscription but billing didn't update",
    "i reduced my plan but still charged the old price"
]

double_charge = [
    "i was charged twice for my subscription this month",
    "i was billed twice this month",
    "i got double charged for my subscription",
    "i was charged two times this month"
]

performance_issues = [
    "overall performance has degraded",
    "the system is much slower than before",
    "the app performance dropped significantly",
    "everything is running slower than usual",
    "performance has decreased noticeably"
]

time_refs = [
    "in the last few days",
    "recently",
    "since the last update",
    "lately",
    "over the past few days"
]

slow_pages = [
    "the data export page",
    "the analytics dashboard page",
    "the notifications page",
    "the api integration page",
    "the login auth page",
    "the billing page",
    "the mobile app page"
]

slow_descriptions = [
    "is very slow",
    "is extremely slow",
    "is taking too long",
    "loads very slowly",
    "is lagging badly",
    "has a significant delay"
]

load_phrases = [
    "and takes a long time to load",
    "and loads very slowly",
    "and takes forever to open",
    "and has a long loading time"
]

ui_sections = [
    "billing section",
    "analytics dashboard section",
    "login auth section",
    "mobile app section",
    "api integration section",
    "notifications section",
    "data export section"
]

crash_actions = [
    "keeps crashing",
    "keeps stopping",
    "keeps closing unexpectedly",
    "stops working",
    "crashes every time"
]

trigger_phrases = [
    "whenever i try to open",
    "when i try to access",
    "every time i open",
    "when accessing"
]

security_requests = [
    "we need details about your data encryption and compliance",
    "please provide information about your security and compliance policies",
    "we require details on data protection and compliance standards",
    "we need clarification on your encryption and compliance practices",
    "please share your data security and compliance details"
]

security_variations = [
    "as part of our internal review",
    "for compliance verification",
    "for security assessment",
    "to meet regulatory requirements",
    "for audit purposes"
]

data_features = [
    "the api integration feature",
    "the login auth feature",
    "the analytics dashboard feature",
    "the mobile app feature",
    "the notifications feature",
    "the billing feature",
    "the data export feature"
]

save_issues = [
    "is not saving my changes",
    "fails to save my changes",
    "does not save my changes",
    "won't save my changes",
    "is unable to save my changes"
]

feature_targets = [
    "api integration",
    "login auth",
    "billing",
    "mobile app",
    "data export",
    "notifications",
    "analytics dashboard"
]

request_phrases = [
    "our team needs more advanced options for",
    "we need more advanced features for",
    "it would be helpful to have more options for",
    "we require additional functionality for"
]

customization_phrases = [
    "can you add an option to customize",
    "is it possible to customize",
    "we would like the ability to customize",
    "please allow customization for"
]

config_targets = [
    "notifications",
    "mobile app",
    "data export",
    "login auth",
    "analytics dashboard",
    "billing",
    "api integration"
]

config_phrases = [
    "can you explain how to configure",
    "how do i properly set up",
    "can you guide me on configuring",
    "i need help configuring",
    "how can i correctly configure"
]

ending_phrases = [
    "properly",
    "correctly",
    "the right way",
    "step by step"
]

account_lock_phrases = [
    "my account was locked after multiple failed login attempts",
    "my account got locked due to too many failed login attempts",
    "i am locked out of my account after several failed logins",
    "my account is locked because of repeated failed login attempts"
]

general_questions = [
    "i have a general question about my account",
    "i need help but i'm not sure which category this fits in",
    "something seems off, can you check my workspace",
    "i have a question regarding my account",
    "i need some general assistance"
]

vulnerability_targets = [
    "notifications functionality",
    "mobile app functionality",
    "billing functionality",
    "analytics dashboard functionality",
    "login auth functionality",
    "api integration functionality",
    "data export functionality"
]

vulnerability_phrases = [
    "there seems to be a vulnerability in",
    "i suspect a security issue in",
    "there might be a vulnerability in",
    "i found a potential vulnerability in"
]

new_feature_targets = [
    "billing",
    "mobile app",
    "notifications",
    "api integration",
    "analytics dashboard",
    "data export",
    "login auth"
]

new_feature_phrases = [
    "it would be great to have",
    "it would be helpful to include",
    "i would like to see",
    "please consider adding"
]

support_suffix = [
    "support in your product",
    "functionality in your platform",
    "capabilities in the system",
    "as a feature in your product"
]

twofa_subjects = [
    "my 2FA code",
    "my two-factor authentication code",
    "the verification code",
    "my authentication code",
    "the 2FA token"
]

twofa_issues = [
    "is not working",
    "isn't being accepted",
    "keeps failing",
    "doesn't work",
    "is invalid",
    "is expired"
]

twofa_contexts = [
    "when I try to sign in",
    "when I log in",
    "during authentication",
    "when accessing my account",
    "while trying to verify"
]

action_types = [
    "export my data from",
    "set up",
    "configure",
    "access",
    "use"
]

features = [
    "notifications",
    "mobile app",
    "api integration",
    "data export",
    "analytics dashboard",
    "billing",
    "login auth"
]

error_subjects = [
    "an error message",
    "a bug",
    "an error",
    "a problem"
]

error_actions = [
    "when I click on",
    "when I try to access",
    "when I open",
    "when I navigate to",
    "while using"
]

error_features = [
    "notifications",
    "analytics dashboard",
    "mobile app",
    "billing",
    "login auth",
    "api integration",
    "data export"
]

error_types = [
    "pops up",
    "appears",
    "shows up",
    "I get an error"
]

In [177]:
def paraphrase_subscription_issue(text, n=3):
    results = set()

    for _ in range(n * 3):
        if "downgrad" in text.lower() or "plan" in text.lower():
            base = random.choice(subscription_actions)
        else:
            base = random.choice(double_charge)

        new_text = random.choice(prefixes) + base + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def paraphrase_performance_crash(text, n=3):
    results = set()

    for _ in range(n * 3):
        issue = random.choice(performance_issues)
        time_ref = random.choice(time_refs)

        new_text = f"{issue} {time_ref}"
        new_text = random.choice(prefixes) + new_text + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def paraphrase_performance_slow(text, n=3):
    results = set()

    for _ in range(n * 3):
        page = random.choice(slow_pages)
        desc = random.choice(slow_descriptions)
        load = random.choice(load_phrases)

        new_text = f"{page} {desc} {load}"
        new_text = random.choice(prefixes) + new_text + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def paraphrase_bug_ui_issue(text, n=3):
    results = set()

    for _ in range(n * 3):
        action = random.choice(crash_actions)
        section = random.choice(ui_sections)
        trigger = random.choice(trigger_phrases)

        new_text = f"the page {action} {trigger} the {section}"
        new_text = random.choice(prefixes) + new_text + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def paraphrase_security_compliance(text, n=3):
    results = set()

    for _ in range(n * 3):
        req = random.choice(security_requests)
        extra = random.choice(security_variations)

        new_text = f"{req} {extra}"
        new_text = random.choice(prefixes) + new_text + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def paraphrase_bug_data_loss(text, n=3):
    results = set()

    for _ in range(n * 3):
        feature = random.choice(data_features)
        issue = random.choice(save_issues)

        new_text = f"{feature} {issue}"
        new_text = random.choice(prefixes) + new_text + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def paraphrase_feature_request(text, n=3):
    results = set()

    for _ in range(n * 3):
        target = random.choice(feature_targets)

        if "custom" in text.lower() or "option" in text.lower():
            base = f"{random.choice(customization_phrases)} {target}"
        else:
            base = f"{random.choice(request_phrases)} {target}"

        new_text = random.choice(prefixes) + base + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def paraphrase_how_to_configuration(text, n=3):
    results = set()

    for _ in range(n * 3):
        target = random.choice(config_targets)
        phrase = random.choice(config_phrases)
        ending = random.choice(ending_phrases)

        new_text = f"{phrase} {target} {ending}"
        new_text = random.choice(prefixes) + new_text + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def paraphrase_account_locked(text, n=3):
    results = set()

    for _ in range(n * 3):
        base = random.choice(account_lock_phrases)

        new_text = random.choice(prefixes) + base + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def paraphrase_general_inquiry(text, n=3):
    results = set()

    for _ in range(n * 3):
        base = random.choice(general_questions)

        new_text = random.choice(prefixes) + base + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def paraphrase_security_vulnerability(text, n=3):
    results = set()

    for _ in range(n * 3):
        target = random.choice(vulnerability_targets)
        phrase = random.choice(vulnerability_phrases)

        base = f"{phrase} {target}"

        new_text = random.choice(prefixes) + base + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def paraphrase_feature_request_new(text, n=3):
    results = set()

    for _ in range(n * 3):
        target = random.choice(new_feature_targets)
        phrase = random.choice(new_feature_phrases)
        suffix = random.choice(support_suffix)

        base = f"{phrase} {target} {suffix}"

        new_text = random.choice(prefixes) + base + random.choice(suffixes)

        results.add(new_text.lower())

        if len(results) >= n:
            break

    return list(results)

def generate_2fa_messages(n=3):
    messages = set()
    
    for _ in range(n * 3):
        subject = random.choice(twofa_subjects)
        issue = random.choice(twofa_issues)
        context = random.choice(twofa_contexts)
        
        base_text = f"{subject} {issue} {context}"
        final_text = random.choice(prefixes) + base_text + random.choice(suffixes)
        
        messages.add(final_text.strip())
        
        if len(messages) >= n:
            break
    
    return list(messages)

def generate_howto_messages(n=3):
    messages = set()
    question_phrases = ["How can I", "How do I", "I need help with", "Can you show me how to"]
    
    for _ in range(n * 3):
        action = random.choice(action_types)
        feature = random.choice(features)
        
        if action == "export my data from":
            base_text = f"how can I {action} the {feature} section?"
        elif action in ["set up", "configure"]:
            base_text = f"I need help {action} {feature} for my team"
        else:
            base_text = f"how do I {action} {feature}?"
        
        final_text = random.choice(prefixes) + base_text + random.choice(suffixes)
        messages.add(final_text.strip())
        
        if len(messages) >= n:
            break
    
    return list(messages)

def generate_bug_messages(n=3):
    messages = set()
    
    for _ in range(n * 3):
        feature = random.choice(error_features)
        error_type = random.choice(error_types)
        
        base_text = f"I am seeing {error_type} when I click on {feature}"
        final_text = random.choice(prefixes) + base_text + random.choice(suffixes)
        
        messages.add(final_text.strip())
        
        if len(messages) >= n:
            break
    
    return list(messages)


In [186]:
augmented_rows = []

for _, row in df_temp2.iterrows():
    text = row['initial_message']
    label = row['refined_issue']


    # if label == "billing_subscription_issue":
    #     new_texts = paraphrase_subscription_issue(text, n=3)

    # elif label == "performance_crash":
    #     new_texts = paraphrase_performance_crash(text, n=3)

    # elif label == "performance_slow":
    #     new_texts = paraphrase_performance_slow(text, n=3)

    # elif label == "bug_ui_issue":
    #     new_texts = paraphrase_bug_ui_issue(text, n=3)

    # elif label == "security_compliance":
    #     new_texts = paraphrase_security_compliance(text, n=3)

    # elif label == "bug_data_loss":
    #     new_texts = paraphrase_bug_data_loss(text, n=3)

    # elif label == "feature_request_improvement":
    #     new_texts = paraphrase_feature_request(text, n=3)
    
    # elif label == "how_to_configuration":
    #     new_texts = paraphrase_how_to_configuration(text, n=3)

    if label == "account_locked":
        new_texts = paraphrase_account_locked(text, n=3)

    elif label == "general_inquiry":
        new_texts = paraphrase_general_inquiry(text, n=3)
    
    # elif label == "security_vulnerability":
    #     new_texts = paraphrase_security_vulnerability(text, n=3)
    
    # elif label == "feature_request_new":
    #     new_texts = paraphrase_feature_request_new(text, n=3)
    elif label == "account_2fa_issue": 
        new_texts = paraphrase_general_inquiry(text, n=3)
    
    # elif label == "how_to_usage":
    #     new_texts = paraphrase_security_vulnerability(text, n=3)
    
    # elif label == "bug_error":
    #     new_texts = paraphrase_feature_request_new(text, n=3)

    else:
        continue

    for t in new_texts:
        augmented_rows.append({
            "initial_message": t,
            "refined_issue": label
        })

df_aug = pd.DataFrame(augmented_rows)
df_final = pd.concat([df_final, df_aug])
df_final.drop_duplicates(subset=['initial_message'], inplace=True)

In [187]:
df_final.shape

(5662, 19)

In [188]:
df_final['refined_issue'].value_counts()

refined_issue
bug_ui_issue                   389
performance_timeout            386
performance_slow               383
how_to_configuration           378
bug_error                      375
account_password_issue         364
bug_data_loss                  354
billing_invoice_issue          348
feature_request_improvement    341
feature_request_new            341
how_to_usage                   326
performance_crash              282
security_compliance            281
security_alert                 259
security_vulnerability         254
billing_subscription_issue     226
general_inquiry                187
account_locked                 144
account_2fa_issue               44
Name: count, dtype: int64

In [190]:
df_final.sample(5)

,created_at,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,platform,region,date,refined_issue
300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"i'm having trouble, performance has decreased noticeably lately since yesterday",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,performance_crash
324,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"there is a problem, operations in the analytics dashboard module is too slow and times out after the last update",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,performance_timeout
539,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"There is a problem, I saw a login I don't recognize from a different location after the last update",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,security_alert
580,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,I detected an unknown login every time I try,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,security_alert
82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,I am not able to access my account; the password is incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,account_password_issue


In [191]:
# df_final.to_csv('Demo.csv',index = False)
# df2_small_balanced.to_csv('small_balanced.csv',index = False)

In [192]:
df_final['initial_message'].nunique()

5662

In [195]:
df_final.duplicated().sum()

0